In [1]:
import numpy as np
import sklearn
import torch
import os
import pandas as pd
import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchmetrics

from transformers import AutoTokenizer

In [2]:
device = 'mps' if torch.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
!wget -q -nc -O IMDB-Dataset.csv "https://www.dropbox.com/scl/fi/0c7zc2adk1mgwgut5w80w/IMDB-Dataset.csv?rlkey=1drfg4zw36mhu32ndy2ihnygw&dl=1"

In [3]:
df = pd.read_csv('IMDB-Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
text = list(df['review'].str.replace('<br />',''))
labels = np.array(df['sentiment'].map({'negative':0,'positive':1}))

In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

Example of how to tokenize text:


In [6]:
seq = text[0][:10]
seq

'One of the'

In [7]:
token_ids = tokenizer(seq)['input_ids']
token_ids

[101, 1448, 1104, 1103, 102]

In [8]:
tokenizer.decode(token_ids+[0,0,0])

'[CLS] One of the [SEP] [PAD] [PAD] [PAD]'

## Dataset Preparation

In [17]:
class TokenDataset(Dataset):
  def __init__(self,tokenized_text,labels,max_seq_len=None):
    self.tokenized_text = tokenized_text
    self.labels = labels
    self.max_seq_len = max_seq_len
    
  def __len__(self):
    return len(self.tokenized_text)

  def __getitem__(self,idx):
    # get requested text
    token_ids = self.tokenized_text[idx]
    label = self.labels[idx]

    # crop or pad as necessary
    if self.max_seq_len is not None:
      if len(token_ids)>self.max_seq_len:
        # choose random substring
        ind = np.random.randint(len(token_ids)-self.max_seq_len)
        token_ids = token_ids[ind:ind+self.max_seq_len]
      else:
        # pad to maximum sequence length
        token_ids = [0]*(self.max_seq_len-len(token_ids)) + token_ids
    # return a sequence of token IDs and a label
    return torch.tensor(token_ids), torch.tensor(label).long()


In [18]:
from sklearn.model_selection import train_test_split

train_text, val_text, train_labels, val_labels = train_test_split(
    text, labels, test_size=0.1, random_state=42, stratify=labels)

train_tokens = tokenizer(train_text)['input_ids']
val_tokens   = tokenizer(val_text)['input_ids']

train_ds = TokenDataset(train_tokens, train_labels, max_seq_len=100)
val_ds   = TokenDataset(val_tokens,   val_labels)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=1)


## 1-D CNN Model

In [19]:
class Transpose(nn.Module):
  def forward(self, x):
    return x.transpose(1, 2)

cnn = nn.Sequential(
    nn.Embedding(tokenizer.vocab_size, 100),
    Transpose(),
    nn.Conv1d(100, 100, kernel_size=3, padding=1), nn.SiLU(),
    nn.Conv1d(100, 100, kernel_size=3, padding=1), nn.SiLU(),
    nn.Conv1d(100, 100, kernel_size=3, padding=1), nn.SiLU(),
    nn.AdaptiveAvgPool1d(1),
    nn.Flatten(),
    nn.Linear(100, 2),
).to(device)


In [20]:
def train(model, train_loader, epochs=10, lr=3e-4):
  opt = torch.optim.Adam(model.parameters(), lr=lr)
  loss_fn = nn.CrossEntropyLoss()
  for epoch in range(epochs):
    model.train()
    total = 0
    for x, y in tqdm.tqdm(train_loader, desc=f'epoch {epoch+1}'):
      x, y = x.to(device), y.to(device)
      loss = loss_fn(model(x), y)
      opt.zero_grad()
      loss.backward()
      opt.step()
      total += loss.item()
    print(f'epoch {epoch+1}: loss {total/len(train_loader):.4f}')

@torch.no_grad()
def evaluate(model, loader):
  model.eval()
  acc = torchmetrics.Accuracy(task='multiclass', num_classes=2).to(device)
  for x, y in tqdm.tqdm(loader):
    x, y = x.to(device), y.to(device)
    acc.update(model(x).argmax(dim=1), y)
  return acc.compute().item()

train(cnn, train_loader)
print('CNN test accuracy:', evaluate(cnn, val_loader))


epoch 1: 100%|██████████| 1407/1407 [00:11<00:00, 124.87it/s]


epoch 1: loss 0.5723


epoch 2: 100%|██████████| 1407/1407 [00:09<00:00, 150.47it/s]


epoch 2: loss 0.4553


epoch 3: 100%|██████████| 1407/1407 [00:10<00:00, 135.96it/s]


epoch 3: loss 0.4080


... (12 outputs left out)

epoch 10: 100%|██████████| 1407/1407 [00:11<00:00, 126.53it/s]


epoch 10: loss 0.2867


100%|██████████| 5000/5000 [00:11<00:00, 420.21it/s]

CNN test accuracy: 0.9074000120162964


In [21]:
class GRUClassifier(nn.Module):
  def __init__(self, vocab_size, embed_dim=100, hidden_dim=100, num_layers=3, num_classes=2):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embed_dim)
    self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
    self.fc = nn.Linear(hidden_dim, num_classes)

  def forward(self, x): 
    x = self.embedding(x)
    out, _ = self.gru(x) 
    logits = self.fc(out)
    return logits[:, -1]

gru = GRUClassifier(tokenizer.vocab_size).to(device)

train(gru, train_loader)
print('GRU test accuracy:', evaluate(gru, val_loader))


epoch 1: 100%|██████████| 1407/1407 [02:17<00:00, 10.27it/s]


epoch 1: loss 0.6227


epoch 2: 100%|██████████| 1407/1407 [02:12<00:00, 10.58it/s]


epoch 2: loss 0.4707


epoch 3: 100%|██████████| 1407/1407 [02:29<00:00,  9.42it/s]


epoch 3: loss 0.4104


... (12 outputs left out)

epoch 10: 100%|██████████| 1407/1407 [02:08<00:00, 10.99it/s]


epoch 10: loss 0.2804


100%|██████████| 5000/5000 [06:25<00:00, 12.98it/s]

GRU test accuracy: 0.894599974155426
